# Task 1: Smart Complementary Product Recommendation Engine
## Step-by-Step Deep Dive

### What are we building?
A system that, given an input product (e.g., Running Shoes), recommends products that are **commonly purchased together** (e.g., Sports Socks, Fitness Watch, Water Bottle).

### Architecture Overview
```
Input Product (image)
    │
    ├──▶ Step 1: Identify product category (e.g., "Running Shoes")
    │
    ├──▶ Step 2: Look up complementary categories (e.g., Socks, Track Pants)
    │
    ├──▶ Step 3: For each complementary category, encode all products with CLIP
    │
    ├──▶ Step 4: Find visually similar products using cosine similarity
    │
    └──▶ Step 5: Rank and return top recommendations
```

---
## Step 0: Install Dependencies & Load Libraries

In [ ]:
!pip install torch torchvision transformers ftfy regex tqdm scikit-learn faiss-cpu pillow matplotlib pandas -q

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
import torch
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity
import faiss
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

---
## Step 1: Load the Fashion Dataset

We use the **Fashion Product Images Small** dataset from Kaggle. It contains:
- `styles.csv` — metadata (product name, category, color, gender, etc.)
- `images/` — product photos

Each product has:
- `id` — unique product ID
- `productDisplayName` — human-readable name
- `articleType` — category (e.g., Running Shoes, Shirts, Jeans)
- `baseColour` — color
- `gender` — Men/Women/Unisex

In [ ]:
# Adjust this path to where your dataset is located
DATA_DIR = "/kaggle/input/fashion-product-images-small"

# Fallback paths
for candidate in ["./fashion-product-images-small", "./data", "../input/fashion-product-images-small"]:
    if os.path.exists(candidate):
        DATA_DIR = candidate
        break

print(f"Data directory: {DATA_DIR}")
print(f"Files: {os.listdir(DATA_DIR)[:10]}")

In [ ]:
# Load the metadata CSV
styles_df = pd.read_csv(os.path.join(DATA_DIR, "styles.csv"), on_bad_lines='skip')
print(f"Raw dataset: {styles_df.shape[0]} rows, {styles_df.shape[1]} columns")
print(f"Columns: {list(styles_df.columns)}")
styles_df.head()

In [ ]:
# Check what product categories exist
print("All unique product types:")
print(styles_df["articleType"].value_counts().to_string())

In [ ]:
# Link each product to its image file on disk
def get_image_path(product_id):
    path = os.path.join(DATA_DIR, "images", f"{product_id}.jpg")
    return path if os.path.exists(path) else None

styles_df["image_path"] = styles_df["id"].apply(get_image_path)

# Keep only products that have an image file
df = styles_df.dropna(subset=["image_path"]).reset_index(drop=True)
print(f"Products with images: {len(df)}")
print(f"Unique categories: {df['articleType'].nunique()}")

# Show distribution
print("\nTop 15 categories by count:")
print(df["articleType"].value_counts().head(15).to_string())

In [ ]:
# Use a manageable subset for faster processing
# Increase SAMPLE_SIZE for better results (e.g., 5000 or all)
SAMPLE_SIZE = 2000

if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"Working with {len(df)} products")
print(f"Categories represented: {df['articleType'].nunique()}")
df[["id", "productDisplayName", "articleType", "baseColour", "gender"]].head(10)

---
## Step 2: Load the CLIP Model

**What is CLIP?**
CLIP (Contrastive Language-Image Pre-training) is a model by OpenAI trained on 400 million image-text pairs.

It has two encoders:
- **Image Encoder** — converts images into 512-dimensional vectors
- **Text Encoder** — converts text into 512-dimensional vectors

Both encoders project into the **same embedding space**, so:
- Similar images are close together
- An image of running shoes is close to the text "running shoes"
- We can measure similarity between any image and text

For Task 1, we use the **image encoder** to compare products visually.

In [ ]:
print("Loading CLIP model (openai/clip-vit-base-patch32)...")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()
print(f"Model loaded! Embedding dimension: {model.config.projection_dim}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## Step 3: Encode All Product Images

This is the **core preprocessing step**. We convert every product image into a 512-dimensional vector using CLIP's image encoder.

### What happens in encoding:
1. Load the image and resize to 224x224
2. Pass through CLIP's Vision Transformer (ViT)
3. Project to 512-dim embedding
4. L2-normalize (so all vectors have unit length)

After encoding, we have:
- `image_embeddings` — numpy array of shape `(num_products, 512)`
- Each row is the "fingerprint" of one product image

In [ ]:
def encode_images(image_paths, batch_size=32):
    """
    Encode images into CLIP embeddings.
    
    For each batch:
    1. Open and convert images to RGB
    2. Preprocess (resize, normalize) using CLIP processor
    3. Forward pass through CLIP image encoder
    4. L2-normalize the output embeddings
    
    Returns:
        all_embeddings: numpy array (n, 512)
        valid_indices: list of original df indices that were successfully encoded
    """
    all_embeddings = []
    valid_indices = []
    
    total = len(image_paths)
    
    for i in range(0, total, batch_size):
        batch_paths = image_paths[i:i+batch_size]
        batch_images = []
        batch_valid = []
        
        # Load images
        for j, path in enumerate(batch_paths):
            try:
                img = Image.open(path).convert("RGB")
                batch_images.append(img)
                batch_valid.append(i + j)
            except Exception as e:
                print(f"  Skipping {path}: {e}")
                continue
        
        if not batch_images:
            continue
        
        # CLIP preprocessing: resize to 224x224, normalize to [0,1], 
        # then apply CLIP-specific normalization (mean/std)
        inputs = processor(
            images=batch_images,
            return_tensors="pt",
            padding=True
        ).to(device)
        
        # Forward pass through CLIP image encoder
        with torch.no_grad():
            embeddings = model.get_image_features(**inputs)
            # L2-normalize: each vector has unit length
            # This makes cosine similarity equivalent to dot product
            embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        
        all_embeddings.append(embeddings.cpu().numpy())
        valid_indices.extend(batch_valid)
        
        processed = min(i + batch_size, total)
        if (i // batch_size) % 10 == 0:
            print(f"  Encoded {processed}/{total} images ({100*processed/total:.0f}%)")
    
    return np.vstack(all_embeddings), valid_indices

print(f"Encoding {len(df)} product images with CLIP...")
image_embeddings, valid_idx = encode_images(df["image_path"].tolist())

# Filter df to only products that were successfully encoded
df = df.iloc[valid_idx].reset_index(drop=True)

print(f"\nDone!")
print(f"Embedding matrix shape: {image_embeddings.shape}")
print(f"  → {image_embeddings.shape[0]} products, {image_embeddings.shape[1]} dimensions each")
print(f"L2 norm of first vector: {np.linalg.norm(image_embeddings[0]):.4f} (should be ~1.0)")

---
## Step 4: Build a Fast Search Index (FAISS)

**Why FAISS?**
When we recommend complementary products, we need to find the most similar product in each complementary category. With 2000+ products, computing similarity on the fly is slow.

**FAISS** (Facebook AI Similarity Search) builds an optimized index that can search millions of vectors in milliseconds.

We use `IndexFlatIP` (Inner Product) which, since our embeddings are L2-normalized, computes exact cosine similarity.

In [ ]:
# Build FAISS index
embedding_dim = image_embeddings.shape[1]  # 512

faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(image_embeddings.astype("float32"))

print(f"FAISS index built:")
print(f"  Total vectors: {faiss_index.ntotal}")
print(f"  Dimension: {faiss_index.d}")
print(f"  Search method: Exact inner product (= cosine similarity for unit vectors)")

---
## Step 5: Define Complementary Category Rules

This is the **domain knowledge** layer. We define which product categories are commonly purchased together.

### Why not just use visual similarity alone?
If we only used visual similarity, recommending "similar" running shoes when someone views running shoes isn't useful. We want **complementary** products — things that go together.

### How the rules work:
- Each key is a product category (e.g., `"Running Shoes"`)
- Each value is a list of complementary categories (e.g., `["Socks", "Track Pants", "Tshirts"]`)
- When a user views a Running Shoe, we search for the best matches in Socks, Track Pants, and Tshirts

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# COMPLEMENTARY CATEGORY RULES
# This encodes fashion domain knowledge about what goes together.
# ═══════════════════════════════════════════════════════════════════════════════

COMPLEMENT_MAP = {
    # ── Athletic / Sports ──────────────────────────────────────────────────────
    "Running Shoes":   ["Socks", "Track Pants", "Tshirts", "Shorts", "Watches"],
    "Sports Shoes":    ["Socks", "Track Pants", "Shorts", "Tshirts"],
    "Sneakers":        ["Jeans", "Tshirts", "Socks", "Hoodies"],
    "Track Pants":     ["Tshirts", "Running Shoes", "Socks", "Sports Shoes"],
    "Sports Sandals":  ["Shorts", "Tshirts"],
    
    # ── Formal / Office ────────────────────────────────────────────────────────
    "Formal Shoes":    ["Trousers", "Shirts", "Belts", "Socks", "Watches"],
    "Shirts":          ["Trousers", "Formal Shoes", "Belts", "Watches", "Jeans"],
    "Trousers":        ["Shirts", "Formal Shoes", "Belts", "Sweaters"],
    "Blazers":         ["Trousers", "Formal Shoes", "Shirts", "Belts"],
    
    # ── Casual ─────────────────────────────────────────────────────────────────
    "Jeans":           ["Tshirts", "Sneakers", "Casual Shoes", "Jackets", "Belts"],
    "Tshirts":         ["Jeans", "Shorts", "Sneakers", "Sunglasses", "Casual Shoes"],
    "Shorts":          ["Tshirts", "Sneakers", "Slippers", "Sports Sandals"],
    "Casual Shoes":    ["Jeans", "Tshirts", "Shorts", "Sneakers"],
    
    # ── Outerwear ──────────────────────────────────────────────────────────────
    "Jackets":         ["Jeans", "Trousers", "Sneakers", "Tshirts", "Shirts"],
    "Hoodies":         ["Jeans", "Sneakers", "Tshirts", "Track Pants"],
    "Sweaters":        ["Trousers", "Jeans", "Formal Shoes", "Shirts"],
    "Rain Jacket":     ["Track Pants", "Sports Shoes"],
    
    # ── Accessories ────────────────────────────────────────────────────────────
    "Watches":         ["Shirts", "Trousers", "Formal Shoes", "Belts", "Blazers"],
    "Belts":           ["Trousers", "Shirts", "Formal Shoes", "Jeans"],
    "Sunglasses":      ["Tshirts", "Shorts", "Jeans", "Casual Shoes"],
    "Bags":            ["Tshirts", "Jeans", "Sneakers", "Hoodies"],
    "Socks":           ["Running Shoes", "Sneakers", "Formal Shoes", "Sports Shoes"],
    
    # ── Women's ────────────────────────────────────────────────────────────────
    "Heels":           ["Dresses", "Kurtas", "Handbags"],
    "Flats":           ["Jeans", "Kurtas", "Tshirts", "Salwar"],
    "Dresses":         ["Heels", "Handbags", "Sunglasses"],
    "Kurtas":          ["Trousers", "Flats", "Earrings", "Salwar"],
    "Salwar":          ["Kurtas", "Flats", "Heels"],
    "Saree":           ["Heels", "Earrings", "Handbags"],
    "Leggings":        ["Kurtas", "Flats", "Tshirts"],
}

# Get all categories present in our dataset
ALL_CATEGORIES = df["articleType"].unique().tolist()

# Count how many categories have rules
covered = sum(1 for c in ALL_CATEGORIES if c in COMPLEMENT_MAP)
print(f"Dataset categories: {len(ALL_CATEGORIES)}")
print(f"Categories with complement rules: {covered}")
print(f"Coverage: {100*covered/len(ALL_CATEGORIES):.0f}%")
print(f"\nRules defined for:")
for cat, comps in COMPLEMENT_MAP.items():
    print(f"  {cat} → {comps}")

---
## Step 6: The Recommendation Algorithm

Here's the complete recommendation pipeline, broken into substeps:

### For a given input product:

**Substep 6a:** Identify the product's category (e.g., "Running Shoes")

**Substep 6b:** Look up complementary categories from `COMPLEMENT_MAP` (e.g., [Socks, Track Pants, Tshirts])

**Substep 6c:** For EACH complementary category:
   - Get all products in that category from the dataset
   - Compute cosine similarity between the input product's embedding and each candidate's embedding
   - Select the top-2 most visually similar products from that category

**Substep 6d:** Pool all candidates across categories

**Substep 6e:** Sort by similarity score, deduplicate, return top-k

In [ ]:
def recommend_complementary(product_idx, top_k=6, verbose=True):
    """
    Recommend complementary products for a given product.
    
    Args:
        product_idx: index into the df DataFrame
        top_k: number of recommendations to return
        verbose: print detailed step-by-step info
    
    Returns:
        dict with input product info and list of recommendations
    """
    product = df.iloc[product_idx]
    product_name = product["productDisplayName"]
    product_category = product["articleType"]
    product_embedding = image_embeddings[product_idx:product_idx+1].astype("float32")
    
    if verbose:
        print(f"=" * 60)
        print(f"INPUT PRODUCT")
        print(f"  Name: {product_name}")
        print(f"  Category: {product_category}")
        print(f"  Color: {product['baseColour']}")
        print(f"  Embedding shape: {product_embedding.shape}")
    
    # ── Substep 6b: Look up complementary categories ──────────────────────
    comp_cats = COMPLEMENT_MAP.get(product_category, [])
    
    if not comp_cats:
        if verbose:
            print(f"\n  No rules for '{product_category}', using random categories...")
        other_cats = [c for c in ALL_CATEGORIES if c != product_category]
        comp_cats = np.random.choice(
            other_cats, size=min(5, len(other_cats)), replace=False
        ).tolist()
    
    if verbose:
        print(f"\n  Complementary categories to search: {comp_cats}")
    
    # ── Substep 6c: For each category, find best visual matches ───────────
    candidates = []
    
    for comp_cat in comp_cats:
        # Find all products in this complementary category
        cat_mask = df["articleType"] == comp_cat
        cat_indices = np.where(cat_mask.values)[0]
        
        if len(cat_indices) == 0:
            if verbose:
                print(f"    [{comp_cat}] No products found, skipping")
            continue
        
        # Compute cosine similarity
        # Since embeddings are L2-normalized, dot product = cosine similarity
        cat_embeddings = image_embeddings[cat_indices].astype("float32")
        similarities = np.dot(cat_embeddings, product_embedding.T).flatten()
        
        # Get top-2 from this category
        top_local_indices = np.argsort(similarities)[-2:][::-1]
        
        if verbose:
            print(f"    [{comp_cat}] {len(cat_indices)} products, "
                  f"best similarity: {similarities[top_local_indices[0]]:.4f}")
        
        for local_idx in top_local_indices:
            global_idx = cat_indices[local_idx]
            candidates.append({
                "idx": int(global_idx),
                "name": df.iloc[global_idx]["productDisplayName"],
                "category": comp_cat,
                "color": df.iloc[global_idx]["baseColour"],
                "similarity": float(similarities[local_idx]),
                "complement_to": product_category,
                "image_path": df.iloc[global_idx]["image_path"]
            })
    
    # ── Substep 6d+6e: Sort, deduplicate, return top-k ────────────────────
    candidates.sort(key=lambda x: x["similarity"], reverse=True)
    
    seen = set()
    unique = []
    for c in candidates:
        if c["idx"] not in seen:
            seen.add(c["idx"])
            unique.append(c)
    
    if verbose:
        print(f"\n  Total candidates: {len(candidates)}, Unique: {len(unique)}")
        print(f"  Returning top {min(top_k, len(unique))}")
        print(f"=" * 60)
    
    return {
        "input_idx": product_idx,
        "input_name": product_name,
        "input_category": product_category,
        "input_image": product["image_path"],
        "input_color": product["baseColour"],
        "recommendations": unique[:top_k]
    }

---
## Step 7: Visualization Function

Display the input product and its recommended complementary products in a grid.

In [ ]:
def visualize_recommendations(result):
    """
    Create a clear visualization:
    - Left: Input product (blue border)
    - Right: Complementary recommendations (green border, ranked by similarity)
    """
    recs = result["recommendations"]
    n_recs = len(recs)
    total_cols = n_recs + 1  # input + recommendations
    
    fig, axes = plt.subplots(1, total_cols, figsize=(4 * total_cols, 5.5))
    
    if total_cols == 1:
        axes = [axes]
    
    # ── Input product ─────────────────────────────────────────────────────
    ax = axes[0]
    img = Image.open(result["input_image"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(
        f"INPUT PRODUCT\n{result['input_name'][:35]}\n"
        f"Category: {result['input_category']}\nColor: {result['input_color']}",
        fontsize=10, fontweight='bold', color='#0A66C2', pad=10
    )
    ax.axis('off')
    # Blue border for input
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#0A66C2')
        spine.set_linewidth(4)
    ax.set_facecolor('#EBF3FB')
    
    # ── Recommendations ───────────────────────────────────────────────────
    for i, rec in enumerate(recs):
        ax = axes[i + 1]
        img = Image.open(rec["image_path"]).convert("RGB")
        ax.imshow(img)
        
        score_pct = rec["similarity"] * 100
        rank_color = '#057642' if i < 3 else '#666'
        
        ax.set_title(
            f"#{i+1} MATCH\n"
            f"{rec['name'][:30]}\n"
            f"Category: {rec['category']}\n"
            f"Similarity: {score_pct:.1f}%",
            fontsize=9, color=rank_color, pad=8
        )
        ax.axis('off')
        # Green border for recommendations
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color('#057642')
            spine.set_linewidth(2)
        ax.set_facecolor('#E8F5EE')
    
    fig.suptitle(
        f"Complementary Recommendations for: {result['input_category']}",
        fontsize=14, fontweight='bold', y=1.03
    )
    
    # Add legend
    blue_patch = mpatches.Patch(color='#0A66C2', label='Input Product')
    green_patch = mpatches.Patch(color='#057642', label='Complementary Recommendation')
    fig.legend(handles=[blue_patch, green_patch], loc='lower center', ncol=2,
              fontsize=10, frameon=True, fancybox=True)
    
    plt.tight_layout(rect=[0, -0.08, 1, 0.98])
    plt.show()

---
## Step 8: Test the Recommendation Engine

Let's test with several different product types to see how the system performs.

In [ ]:
# First, let's see what products are available in key categories
print("Available products by category:\n")
for cat in ["Running Shoes", "Shirts", "Jeans", "Watches", "Tshirts"]:
    subset = df[df["articleType"] == cat]
    if len(subset) > 0:
        print(f"  {cat} ({len(subset)} products):")
        for _, row in subset.head(3).iterrows():
            print(f"    - idx={row.name}: {row['productDisplayName']} ({row['baseColour']})")
        print()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEST 1: Running Shoe
# Expected complementary: Sports Socks, Track Pants, Tshirts, etc.
# ═══════════════════════════════════════════════════════════════════════════════

running_mask = df["articleType"] == "Running Shoes"
if running_mask.sum() > 0:
    test_idx = df[running_mask].index[0]
    
    print("STEP-BY-STEP WALKTHROUGH:")
    print("=" * 60)
    result = recommend_complementary(test_idx, top_k=6, verbose=True)
    
    print("\nFINAL RECOMMENDATIONS:")
    print("-" * 60)
    for i, rec in enumerate(result['recommendations']):
        print(f"  {i+1}. [{rec['category']}] {rec['name']}")
        print(f"     Color: {rec['color']} | Similarity: {rec['similarity']:.4f}")
    
    visualize_recommendations(result)
else:
    print("No Running Shoes in sample, trying another category...")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEST 2: Formal Shirt
# Expected: Trousers, Formal Shoes, Belts, Watches
# ═══════════════════════════════════════════════════════════════════════════════

shirt_mask = df["articleType"] == "Shirts"
if shirt_mask.sum() > 0:
    test_idx = df[shirt_mask].index[0]
    result = recommend_complementary(test_idx, top_k=6, verbose=True)
    
    print("\nFINAL RECOMMENDATIONS:")
    print("-" * 60)
    for i, rec in enumerate(result['recommendations']):
        print(f"  {i+1}. [{rec['category']}] {rec['name']} — sim: {rec['similarity']:.4f}")
    
    visualize_recommendations(result)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEST 3: Jeans
# Expected: Tshirts, Sneakers, Casual Shoes, Jackets
# ═══════════════════════════════════════════════════════════════════════════════

jeans_mask = df["articleType"] == "Jeans"
if jeans_mask.sum() > 0:
    test_idx = df[jeans_mask].index[0]
    result = recommend_complementary(test_idx, top_k=6, verbose=True)
    
    print("\nFINAL RECOMMENDATIONS:")
    print("-" * 60)
    for i, rec in enumerate(result['recommendations']):
        print(f"  {i+1}. [{rec['category']}] {rec['name']} — sim: {rec['similarity']:.4f}")
    
    visualize_recommendations(result)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEST 4: Sneakers (casual)
# Expected: Jeans, Tshirts, Socks, Hoodies
# ═══════════════════════════════════════════════════════════════════════════════

sneaker_mask = df["articleType"] == "Sneakers"
if sneaker_mask.sum() > 0:
    test_idx = df[sneaker_mask].index[0]
    result = recommend_complementary(test_idx, top_k=6, verbose=True)
    
    print("\nFINAL RECOMMENDATIONS:")
    print("-" * 60)
    for i, rec in enumerate(result['recommendations']):
        print(f"  {i+1}. [{rec['category']}] {rec['name']} — sim: {rec['similarity']:.4f}")
    
    visualize_recommendations(result)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TEST 5: Watch
# Expected: Shirts, Trousers, Formal Shoes, Belts, Blazers
# ═══════════════════════════════════════════════════════════════════════════════

watch_mask = df["articleType"] == "Watches"
if watch_mask.sum() > 0:
    test_idx = df[watch_mask].index[0]
    result = recommend_complementary(test_idx, top_k=6, verbose=True)
    
    print("\nFINAL RECOMMENDATIONS:")
    print("-" * 60)
    for i, rec in enumerate(result['recommendations']):
        print(f"  {i+1}. [{rec['category']}] {rec['name']} — sim: {rec['similarity']:.4f}")
    
    visualize_recommendations(result)

---
## Step 9: Understanding the Similarity Scores

In [ ]:
# Let's analyze what the similarity scores mean
# Pick a product and show similarity to various categories

test_idx = df[df["articleType"] == "Running Shoes"].index[0]
test_emb = image_embeddings[test_idx:test_idx+1].astype("float32")

print(f"Input: {df.iloc[test_idx]['productDisplayName']}")
print(f"Category: {df.iloc[test_idx]['articleType']}")
print("\nSimilarity distribution across ALL categories:")
print("-" * 55)

cat_similarities = {}
for cat in sorted(df["articleType"].unique()):
    cat_mask = df["articleType"] == cat
    cat_indices = np.where(cat_mask.values)[0]
    if len(cat_indices) == 0:
        continue
    sims = np.dot(image_embeddings[cat_indices].astype("float32"), test_emb.T).flatten()
    cat_similarities[cat] = {
        "mean": float(np.mean(sims)),
        "max": float(np.max(sims)),
        "count": len(cat_indices)
    }

# Sort by mean similarity
sorted_cats = sorted(cat_similarities.items(), key=lambda x: x[1]["mean"], reverse=True)

print(f"{'Category':<25} {'Mean Sim':>10} {'Max Sim':>10} {'Count':>6}")
print("-" * 55)
for cat, stats in sorted_cats[:20]:
    bar = "█" * int(stats["mean"] * 50)
    print(f"{cat:<25} {stats['mean']:>10.4f} {stats['max']:>10.4f} {stats['count']:>6}  {bar}")

print("\n...")
print("\nLowest similarity (least related):")
for cat, stats in sorted_cats[-5:]:
    print(f"{cat:<25} {stats['mean']:>10.4f} {stats['max']:>10.4f} {stats['count']:>6}")

In [ ]:
# Visualize the category similarity heatmap
cats_to_show = [c for c, _ in sorted_cats[:15]]
means = [cat_similarities[c]["mean"] for c in cats_to_show]
maxes = [cat_similarities[c]["max"] for c in cats_to_show]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(cats_to_show))
width = 0.35

bars1 = ax.bar(x - width/2, means, width, label='Mean Similarity', color='#0A66C2', alpha=0.8)
bars2 = ax.bar(x + width/2, maxes, width, label='Max Similarity', color='#057642', alpha=0.8)

ax.set_xlabel('Product Category', fontsize=12)
ax.set_ylabel('Cosine Similarity', fontsize=12)
ax.set_title(f'Visual Similarity to: {df.iloc[test_idx]["productDisplayName"][:40]}',
            fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(cats_to_show, rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=10)
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Step 10: Explanation Summary

### How Recommendations Are Generated

| Step | What Happens | Technology |
|------|-------------|------------|
| 1 | User views a product (e.g., Running Shoe) | Chrome Extension / UI |
| 2 | System identifies the product's category | Dataset metadata |
| 3 | Look up complementary categories | Curated `COMPLEMENT_MAP` rules |
| 4 | For each complementary category, find all products | Pandas filtering |
| 5 | Compute visual similarity to input product | CLIP image embeddings + dot product |
| 6 | Select top matches per category | NumPy argsort |
| 7 | Rank all candidates globally, deduplicate | Sort by similarity score |
| 8 | Display top-k recommendations | Matplotlib visualization |

### Why This Approach Works

1. **Domain Knowledge + Data** — The complement map encodes fashion expertise (what goes together), while CLIP ensures the recommended products are visually appropriate (matching style/color).

2. **Cross-Category Search** — We don't just find visually similar products (that would give more shoes). We search in complementary categories (socks, pants, accessories).

3. **Style-Aware** — Within each complementary category, CLIP embeddings ensure we pick items that visually match the input. A running shoe gets sporty socks, not dress socks.

4. **Scalable** — FAISS index enables searching millions of products in milliseconds.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# BONUS: Interactive recommendation demo
# Try different products by changing the category and index
# ═══════════════════════════════════════════════════════════════════════════════

# Pick random products from different categories for a gallery view
categories_to_demo = ["Running Shoes", "Shirts", "Jeans", "Watches"]
fig, axes = plt.subplots(1, len(categories_to_demo), figsize=(4 * len(categories_to_demo), 4))

for i, cat in enumerate(categories_to_demo):
    subset = df[df["articleType"] == cat]
    if len(subset) > 0:
        row = subset.iloc[0]
        img = Image.open(row["image_path"]).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(f"{cat}\n{row['productDisplayName'][:25]}", fontsize=10)
    axes[i].axis('off')

fig.suptitle("Try These Products as Inputs", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nTo test a different product, run:")
print("  result = recommend_complementary(product_idx, top_k=6)")
print("  visualize_recommendations(result)")